In [60]:
from langgraph.graph import StateGraph, START, END, MessagesState

In [61]:
class AgentState(MessagesState):pass

In [62]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")
else:
    print("✅ Key imported successfully!")

✅ Key imported successfully!


In [63]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # or "gemini-1.5-pro"
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)


In [64]:
def add(a:int, b:int)-> int:
    """ This function takes 2 agrs a and b , add two args and return"""
    return a+b

In [65]:
llm_with_tools = llm.bind_tools([add])

In [66]:
llm_with_tools

RunnableBinding(bound=ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x7613fcc6be30>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'add', 'description': 'This function takes 2 agrs a and b , add two args and return', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [67]:
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage, SystemMessage

In [68]:
# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs")

In [69]:
def tool_calling_llm(state: AgentState): 
    return {"messages": [llm_with_tools.invoke( [sys_msg] + state["messages"]) ]}

In [70]:
graph = StateGraph(AgentState)

graph.add_node("tool_call", tool_calling_llm)
graph.add_node("tools", ToolNode([add]))


graph.add_edge(START, "tool_call")
graph.add_conditional_edges(
    "tool_call",
    tools_condition
)

graph.add_edge("tools",END)

In [71]:


app = graph.compile()

In [72]:
for event in app.stream({"messages":[HumanMessage(content="give me latest ai news")]}):
    for value in event.values():
        print(value["messages"][-1].content)

I am sorry, I cannot provide you with the latest AI news. My capabilities are limited to performing arithmetic operations like addition.


In [73]:
response = app.invoke({"messages": [HumanMessage(content="add two number 111 and 34343222 ")]})


In [74]:
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

add two number 111 and 34343222 
================================== Ai Message ==================================
Tool Calls:
  add (13733e85-0aa9-4c69-9d3e-064588d1c827)
 Call ID: 13733e85-0aa9-4c69-9d3e-064588d1c827
  Args:
    a: 111.0
    b: 34343222.0
================================= Tool Message =================================
Name: add

34343333
